# Worked Capstone: Production Churn Service

**Domain:** MLOps and system engineering  
**Primary dataset:** `customer_churn.csv`  
**Level:** Practitioner to Advanced

## Business goal

Train, serialize, validate, serve, test, and monitor a complete churn pipeline using the bundled production package.

This is a worked reference project. First attempt the corresponding phase project independently; then use this capstone to compare framing, evaluation, code structure, and communication.

## Decision questions

        1. Can the artifact be reproduced?
2. Are training and serving features identical?
3. Does the API reject invalid input?
4. What is monitored and how is rollback triggered?

        ## Definition of done

        - [ ] Training artifact
- [ ] Inference parity check
- [ ] API contract
- [ ] API test
- [ ] Drift example
- [ ] Release gates
- [ ] Model card

## End-to-end workflow

```text
Decision and scope
      ↓
Data contract and quality
      ↓
Exploration and hypotheses
      ↓
Baseline and evaluation design
      ↓
Candidate method(s)
      ↓
Held-out / temporal evaluation
      ↓
Error, slice, and sensitivity analysis
      ↓
Artifacts, limitations, recommendation
```

At every stage, distinguish calculation correctness, statistical validity, operational validity, and decision validity.

## Risk register

        | Risk | Mitigation |
        |---|---|
        | Artifact loaded with incompatible code | Pin versions and run compatibility tests. |
| Feature contract divergence | Share feature definitions and validate at boundaries. |
| Silent model degradation | Layer monitoring and outcome capture. |
| Unsafe automatic decisions | Use stated scope, thresholds, review, and abstention. |

In [ ]:
from pathlib import Path
import sys
import json
import warnings
warnings.filterwarnings("ignore")

_candidates = [Path.cwd(), *Path.cwd().parents]
COURSE_ROOT = next((p for p in _candidates if (p / "datasets").exists()), Path.cwd())
DATA_DIR = COURSE_ROOT / "datasets"
ARTIFACT_DIR = COURSE_ROOT / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)
sys.path.insert(0, str(COURSE_ROOT))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print(f"Course root: {COURSE_ROOT}")

## 1. Train package artifact

Use the importable production example rather than duplicating notebook logic.

In [ ]:
from src.production_example.config import Settings
from src.production_example.train import train
from src.production_example.predict import predict_records
import joblib

settings=Settings(model_path=ARTIFACT_DIR/"production_churn_pipeline.joblib",decision_threshold=.40)
metrics=train(DATA_DIR/"customer_churn.csv",settings)
print(metrics)
assert settings.model_path.exists()

## 2. Inference parity and contract

Predict through the same feature definitions and serialized preprocessing used in training.

In [ ]:
records=pd.read_csv(DATA_DIR/"customer_churn.csv").drop(columns=["churn"]).head(5)
output=predict_records(records,settings)
display(output[["customer_id","churn_probability","churn_prediction"]])
assert output.churn_probability.between(0,1).all()

## 3. API contract test

Override the demonstration API settings to use the freshly trained artifact, then test health, valid input, and invalid input.

In [ ]:
from fastapi.testclient import TestClient
import src.production_example.api as api_module

api_module.settings=settings
api_module.get_model.cache_clear()
request={
    "tenure_months":12,"monthly_charges":85.0,"support_tickets_90d":2,
    "weekly_usage_hours":20.0,"contract_type":"Month-to-month",
    "internet_service":"Fiber","autopay":0,"senior_citizen":0
}
invalid={**request,"tenure_months":-1}
with TestClient(api_module.app) as client:
    health_response=client.get("/health")
    valid_response=client.post("/predict",json=request)
    invalid_response=client.post("/predict",json=invalid)
print(health_response.json())
print(valid_response.status_code,valid_response.json())
print("Invalid request status:",invalid_response.status_code)
assert health_response.status_code==200
assert valid_response.status_code==200
assert invalid_response.status_code==422

## 4. Monitoring and drift

Drift is a review signal. Combine it with service, score, outcome, and subgroup evidence.

In [ ]:
from src.course_utils import population_stability_index
training=pd.read_csv(DATA_DIR/"customer_churn.csv")
rng_local=np.random.default_rng(42)
simulated_current=training.monthly_charges.to_numpy()+rng_local.normal(12,5,len(training))
psi=population_stability_index(training.monthly_charges,simulated_current)
monitoring=pd.DataFrame([
    ["service","p50/p95/p99 latency, throughput, 4xx/5xx, saturation"],
    ["data","schema failures, missingness, ranges, categories, PSI"],
    ["model","score distribution, calibration, slice performance"],
    ["decision","selection, overrides, abstention, intervention outcome"],
],columns=["layer","signals"])
display(monitoring)
print("Simulated monthly-charge PSI:",psi)

## 5. Release gates and rollback

A release is an approved system change, not merely a model upload.

In [ ]:
gates=pd.DataFrame([
    ["Reproducible training","pass",str(settings.model_path)],
    ["Unit/contract tests","required","pytest -q"],
    ["Offline metric comparison","pass",json.dumps(metrics)],
    ["Security/dependency scan","required","CI/CD"],
    ["Canary SLO and metric gates","required","release plan"],
    ["Rollback artifact","required","previous immutable version"],
    ["Owner and on-call playbook","required","operational document"],
],columns=["gate","status","evidence"])
display(gates)
gates.to_csv(ARTIFACT_DIR/"production_release_gates.csv",index=False)

## Model/project card

Complete this before presenting the result:

| Field | Statement |
|---|---|
| Intended use | |
| Excluded use | |
| Data population and coverage | |
| Target/metric definition | |
| Evaluation split | |
| Baseline | |
| Primary result | |
| Known limitations | |
| Important subgroup behaviour | |
| Human review / abstention | |
| Monitoring | |
| Owner and review cadence | |

## Final reflection

1. Which result changed your initial belief?
2. Which assumption creates the largest residual risk?
3. What simpler alternative was competitive?
4. What evidence is still required before an operational decision?
5. What would you monitor first after release?

Re-run the notebook from a clean kernel and verify generated artifacts before considering the capstone complete.